# SWYNEX Internship — Task 1: Data Preparation

**Dataset:** Palmer Penguins (public dataset)

**Objective:** Inspect a public dataset, clean missing values, standardize data types, document assumptions, and export a cleaned CSV.

Source CSV: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv
Dataset contains penguin species, island, bill/flipper measurements, body mass, sex, and year.

In [4]:
import pandas as pd
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_URL)
print("Original shape:", df.shape)
display(df.head())

Original shape: (344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


## 1. Inspect structure and quality

In [5]:
print("Column names:\n", df.columns.tolist())
print("\nData types and non-null counts:")
df.info()
print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\nExact duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

Column names:
 ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']

Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 18.9 KB

Missing values per column:


,missing_count
sex,11
bill_depth_mm,2
bill_length_mm,2
flipper_length_mm,2
body_mass_g,2
island,0
species,0



Exact duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
species,344,3,Adelie,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
island,344,3,Biscoe,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bill_length_mm,342.0,NaN,NaN,NaN,43.92193,5.459584,32.1,39.225,44.45,48.5,59.6
bill_depth_mm,342.0,NaN,NaN,NaN,17.15117,1.974793,13.1,15.6,17.3,18.7,21.5
flipper_length_mm,342.0,NaN,NaN,NaN,200.915205,14.061714,172.0,190.0,197.0,213.0,231.0
body_mass_g,342.0,NaN,NaN,NaN,4201.754386,801.954536,2700.0,3550.0,4050.0,4750.0,6300.0
sex,333,2,MALE,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Clean column names, values, duplicates, and types

Missing numeric measurements are imputed with the median of their respective columns. Missing categorical values are imputed with the mode (most frequent value). This keeps rows available for basic analysis, but the imputed values should not be treated as observed measurements.

In [7]:

clean = df.copy()

# Clean column names and text values
clean.columns = [c.strip().lower().replace(" ", "_") for c in clean.columns]

for col in clean.select_dtypes(include=["str", "object"]).columns:
    clean[col] = clean[col].str.strip()
    clean[col] = clean[col].replace("", pd.NA)

# Remove exact duplicate rows
before_duplicates = len(clean)
clean = clean.drop_duplicates().copy()
duplicates_removed = before_duplicates - len(clean)

# Correct numeric columns (this dataset has no year column)
numeric_cols = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g"
]

for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Fill missing numeric values with column medians
numeric_imputation = {}

for col in numeric_cols:
    if clean[col].isna().any():
        median_value = clean[col].median()
        numeric_imputation[col] = float(median_value)
        clean[col] = clean[col].fillna(median_value)

# Fill missing categorical values with the mode
categorical_cols = ["species", "island", "sex"]
categorical_imputation = {}

for col in categorical_cols:
    if clean[col].isna().any():
        mode_values = clean[col].mode(dropna=True)
        if not mode_values.empty:
            mode_value = mode_values.iloc[0]
            categorical_imputation[col] = str(mode_value)
            clean[col] = clean[col].fillna(mode_value)

# Set appropriate data types
for col in categorical_cols:
    clean[col] = clean[col].astype("category")

print("Duplicates removed:", duplicates_removed)
print("Numeric imputation values:", numeric_imputation)
print("Categorical imputation values:", categorical_imputation)
print("Remaining missing values:", int(clean.isna().sum().sum()))
print("Cleaned shape:", clean.shape)

display(clean.head())


Duplicates removed: 0
Numeric imputation values: {'bill_length_mm': 44.45, 'bill_depth_mm': 17.3, 'flipper_length_mm': 197.0, 'body_mass_g': 4050.0}
Categorical imputation values: {'sex': 'MALE'}
Remaining missing values: 0
Cleaned shape: (344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.10,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.50,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.30,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,44.45,17.3,197.0,4050.0,MALE
4,Adelie,Torgersen,36.70,19.3,193.0,3450.0,FEMALE


## 3. Validate the cleaned dataset

In [8]:
validation = pd.DataFrame({
    "original_dtype": df.dtypes.astype(str),
    "cleaned_dtype": clean.dtypes.astype(str),
    "missing_before": df.isna().sum(),
    "missing_after": clean.isna().sum()
})
display(validation)

assert clean.isna().sum().sum() == 0, "Missing values remain."
assert clean.duplicated().sum() == 0, "Duplicate rows remain."
assert clean[ numeric_cols ].select_dtypes(include="number").shape[1] == len(numeric_cols)
print("Validation passed: no missing values or duplicate rows remain.")

,original_dtype,cleaned_dtype,missing_before,missing_after
species,str,category,0,0
island,str,category,0,0
bill_length_mm,float64,float64,2,0
bill_depth_mm,float64,float64,2,0
flipper_length_mm,float64,float64,2,0
body_mass_g,float64,float64,2,0
sex,str,category,11,0


Validation passed: no missing values or duplicate rows remain.


## 4. Export the cleaned dataset

In [9]:
output_path = OUTPUT_DIR / "penguins_cleaned.csv"
clean.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")

Saved cleaned dataset to: ..\data\processed\penguins_cleaned.csv


## 5. Assumptions and limitations

- Blank strings are treated as missing values after trimming whitespace.
- Numeric fields are converted with `errors='coerce'`; unparseable values become missing and are then imputed.
- Numeric missing values are filled using each column's median, which is less sensitive to extreme values than the mean.
- Categorical missing values are filled using each column's most frequent category. This may increase the apparent frequency of that category.
- Exact duplicate rows are removed; records that merely look similar are not assumed to be duplicates.
- No outliers are removed because unusual measurements may be valid observations.
- This is a data-cleaning task only; no predictive model is trained.
- The source dataset is downloaded from a public repository at runtime, so an internet connection is needed for the first cell.